# Capstone 1: Requirements-to-Test-Cases Pipeline

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build an end-to-end **Requirements-to-Test-Cases Pipeline** that takes raw requirements, validates them, generates user stories, produces test cases, creates test data, and outputs a complete test plan -- all orchestrated by LLMs.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Completed Chapters 1-15


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## Pipeline Architecture

The pipeline has 5 stages:
1. **Validate** -- Check requirement quality
2. **Decompose** -- Break into user stories
3. **Generate Tests** -- Create test cases from stories
4. **Create Test Data** -- Generate data for each test
5. **Assemble Plan** -- Produce the final test plan document


In [ ]:
# Input: raw requirements
requirements = [
    {
        'id': 'REQ-001',
        'text': 'The system shall allow registered users to book appointments with available doctors. Users must select a specialty, choose a doctor, pick an available time slot, and confirm the booking. Email and SMS confirmations shall be sent.'
    },
    {
        'id': 'REQ-002',
        'text': 'The system shall allow users to cancel appointments up to 4 hours before the scheduled time. Cancellations within 4 hours incur a $25 fee. The cancelled slot shall be made available to other users immediately.'
    },
    {
        'id': 'REQ-003',
        'text': 'The system shall display a dashboard showing upcoming appointments, past visit history, and pending prescriptions. The dashboard shall load within 2 seconds.'
    }
]

print(f'Pipeline input: {len(requirements)} requirements')

## Stage 1: Requirement Validation


In [ ]:
def validate_requirements(reqs: list) -> list:
    prompt = f"""Validate each requirement for quality. Check ambiguity, completeness, testability.

Requirements: {json.dumps(reqs)}

For each, return: id, quality_score (1-10), issues (list), improved_text.
Only include improved_text if quality_score < 7.
Return a JSON array. Return ONLY valid JSON."""
    response = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0.2
    )
    return json.loads(response.choices[0].message.content)

validated = validate_requirements(requirements)
print('Stage 1: Validation Complete')
for v in validated:
    print(f"  {v['id']}: Score {v['quality_score']}/10 | Issues: {len(v.get('issues', []))}")

## Stage 2: Decompose into User Stories


In [ ]:
def decompose_to_stories(reqs: list) -> list:
    prompt = f"""Break each requirement into user stories with acceptance criteria.

Requirements: {json.dumps(reqs)}

For each requirement, generate 2-3 user stories. Each story has:
- story_id, source_req_id, title, as_a, i_want, so_that
- acceptance_criteria: list of Given/When/Then
- story_points

Return a JSON array of all stories. Return ONLY valid JSON."""
    response = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0.3
    )
    return json.loads(response.choices[0].message.content)

stories = decompose_to_stories(requirements)
print(f'Stage 2: Generated {len(stories)} user stories')
for s in stories:
    print(f"  {s['story_id']} (from {s['source_req_id']}): {s['title']}")

## Stage 3: Generate Test Cases


In [ ]:
def generate_tests_from_stories(stories: list) -> list:
    prompt = f"""Generate test cases for each user story. Include positive, negative, and boundary tests.

Stories: {json.dumps(stories)}

For each story, generate 3-5 test cases with:
- tc_id, story_id, title, type (positive/negative/boundary/edge_case)
- preconditions, steps (list), expected_result, priority, test_data

Return a JSON array. Return ONLY valid JSON."""
    response = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': prompt}], temperature=0.2
    )
    return json.loads(response.choices[0].message.content)

test_cases = generate_tests_from_stories(stories)
print(f'Stage 3: Generated {len(test_cases)} test cases')
from collections import Counter
type_counts = Counter(tc.get('type', 'unknown') for tc in test_cases)
for t, c in type_counts.most_common():
    print(f'  {t}: {c}')

## Stage 4 & 5: Test Data and Final Plan


In [ ]:
# Stage 5: Assemble the complete test plan
plan_prompt = f"""Create a comprehensive test plan document from these artifacts.

Requirements: {json.dumps(requirements)}
Validation Results: {json.dumps(validated)}
User Stories: {json.dumps(stories)}
Test Cases: {json.dumps(test_cases)}

The test plan should include:
1. Introduction and Scope
2. Test Strategy (types of testing, entry/exit criteria)
3. Requirements Traceability Matrix (REQ -> Story -> Test Cases)
4. Test Case Summary by priority and type
5. Test Environment Requirements
6. Execution Schedule
7. Risk Assessment
8. Exit Criteria

Format as a professional test plan document."""

response = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': plan_prompt}], temperature=0.3
)
print('Stage 5: TEST PLAN')
print('=' * 60)
print(response.choices[0].message.content)

In [ ]:
# Pipeline summary
print('\nPIPELINE SUMMARY')
print('=' * 40)
print(f'Requirements processed: {len(requirements)}')
print(f'Quality issues found:   {sum(len(v.get("issues", [])) for v in validated)}')
print(f'User stories created:   {len(stories)}')
print(f'Test cases generated:   {len(test_cases)}')
print(f'\nTraceability:')
for req in requirements:
    req_stories = [s for s in stories if s.get('source_req_id') == req['id']]
    req_tests = [tc for tc in test_cases if tc.get('story_id') in [s['story_id'] for s in req_stories]]
    print(f"  {req['id']} -> {len(req_stories)} stories -> {len(req_tests)} test cases")

## Exercises
1. Add error handling and retry logic to each pipeline stage.
2. Export the complete pipeline output to a structured HTML or PDF report.
3. Add a feedback loop where evaluation results from Chapter 15 trigger automatic improvements.
4. Integrate with Jira API to create stories and test cases directly in your project management tool.
